# Chapter 15: Design Google Drive
*From "System Design Interview"*

## TL;DR

Design a cloud file storage and sync service like Google Drive / Dropbox. Files are split into **blocks** (~4 MB), each compressed, encrypted, and stored independently in cloud storage. Only **modified blocks** are synced (delta sync) to minimize bandwidth. A **notification service** (long polling) tells clients about remote changes. **Strong consistency** is enforced via a relational metadata DB with ACID guarantees, and storage costs are managed through deduplication, version limits, and cold storage.

## Requirements

### Functional
- Upload and download files (up to 10 GB)
- Sync files across multiple devices
- File revision history
- Share files with others
- Notifications on file changes

### Non-Functional
- **Reliability**: zero data loss
- **Fast sync speed**
- **Low bandwidth usage**
- **Scalable** and **highly available**
- 50M signed-up users, 10M DAU

## Back-of-the-Envelope Estimation

In [ ]:
# Google Drive estimation
signed_up_users = 50_000_000
dau = 10_000_000
free_space_gb = 10
uploads_per_day = 2
avg_file_size_kb = 500

# Total allocated space
total_space_pb = signed_up_users * free_space_gb / 1e6
print(f"Total allocated space:  {total_space_pb:.0f} PB")

# QPS for uploads
upload_qps = dau * uploads_per_day / (24 * 3600)
peak_upload_qps = upload_qps * 2
print(f"Upload QPS:            {upload_qps:.0f}")
print(f"Peak upload QPS:       {peak_upload_qps:.0f}")

# Daily upload volume
daily_upload_gb = dau * uploads_per_day * avg_file_size_kb / 1e6
print(f"Daily upload volume:   {daily_upload_gb:.0f} GB")

# Read:write ratio is 1:1
print(f"Read:Write ratio:      1:1")

## High-Level Design

```mermaid
graph TB
    C1[Client 1] & C2[Client 2]

    C1 & C2 --> LB[Load Balancer]

    LB --> API[API Servers]
    LB --> BS[Block Servers]

    API --> MDB[(Metadata DB)]
    API --> MC[Metadata Cache]
    BS --> CS[(Cloud Storage S3)]
    CS --> COLD[(Cold Storage S3 Glacier)]

    NS[Notification Service Long Polling] --> C1
    NS --> C2
    OBQ[Offline Backup Queue] --> NS
```

| Component | Purpose |
|-----------|---------|
| **Block Servers** | Split, compress, encrypt, upload/download file blocks |
| **Cloud Storage (S3)** | Store file blocks with cross-region replication |
| **Metadata DB** | Relational DB for files, versions, blocks, users |
| **Notification Service** | Long polling to push file-change events to clients |
| **Offline Backup Queue** | Queue changes for clients that are offline |

## Deep Dive: Block Servers and Delta Sync

### Block Processing Pipeline

```mermaid
graph LR
    F[File] --> SP[Split into 4 MB blocks]
    SP --> CP[Compress gzip/bzip2]
    CP --> EN[Encrypt]
    EN --> UP[Upload to Cloud Storage]
```

### Delta Sync
When a file is edited, **only changed blocks** are uploaded:
- Each block has a unique **hash value**
- System compares hashes to detect which blocks changed
- Unchanged blocks are skipped entirely

**Example**: A 100 MB file with 25 blocks. User edits content in blocks 2 and 5. Only those 2 blocks (8 MB) are uploaded instead of the full 100 MB.

## Deep Dive: File Sync and Conflict Resolution

### Upload Flow
1. Client sends file metadata to API servers (status: "pending")
2. Client uploads blocks to block servers in parallel
3. Block servers compress, encrypt, store in cloud
4. On completion, status updated to "uploaded"
5. Notification service alerts other clients

### Download Flow
1. Notification service informs client of remote change
2. Client fetches updated metadata from API
3. Client downloads only new/changed blocks from block servers
4. Client reconstructs file locally

### Conflict Resolution
- **First write wins**: the first-processed version is accepted
- The second user gets a **sync conflict** with both copies
- User manually merges or chooses which version to keep

## Deep Dive: Metadata Database

### Schema (simplified)

| Table | Key Fields |
|-------|-----------|
| **User** | user_id, username, email |
| **Device** | device_id, user_id, push_id |
| **Namespace** | namespace_id, user_id (root directory) |
| **File** | file_id, name, namespace_id, is_dir, latest_version |
| **File_Version** | version_id, file_id, size, checksum (append-only) |
| **Block** | block_id, file_version_id, block_order, hash, storage_path |

### Why Relational DB?
- **ACID guarantees** ensure strong consistency
- All clients see the same file state at any point
- Cache invalidation on every write keeps cache consistent

## Deep Dive: Notification Service

**Choice: Long Polling** (not WebSocket)

| Factor | Long Polling | WebSocket |
|--------|-------------|-----------|
| Direction | Server-to-client (sufficient) | Bidirectional (overkill) |
| Frequency | Infrequent notifications | Real-time continuous |
| Use case fit | File sync notifications | Chat apps |
| Dropbox uses | Yes | No |

**Flow**:
1. Client opens long-poll connection
2. Server holds connection until a file change occurs (or timeout)
3. On event, server sends response, client processes it
4. Client immediately reopens a new long-poll connection

## Deep Dive: Storage Space Optimization

| Technique | How It Works |
|-----------|-------------|
| **Block deduplication** | Blocks with identical hashes stored once; shared by reference |
| **Version limits** | Cap max versions per file; drop oldest when exceeded |
| **Intelligent pruning** | Weight recent versions; reduce saves for rapidly-edited files |
| **Cold storage** | Move inactive data to S3 Glacier (much cheaper than standard S3) |

## Trade-offs

| Decision | Pro | Con |
|----------|-----|-----|
| Block-level storage (4 MB blocks) | Delta sync, dedup, parallelism | Complexity in block management |
| Long polling (vs WebSocket) | Simpler, sufficient for file sync | Not suitable if real-time bidirectional needed |
| Relational DB for metadata | Strong consistency (ACID) | Less flexible schema, harder to shard |
| Server-side block processing | Centralized logic, easier to secure | Server becomes bottleneck |
| Cross-region S3 replication | Durability, disaster recovery | Higher storage cost, replication lag |

## Key Takeaways

1. **Block-level storage with delta sync** is the key to minimizing bandwidth for file sync
2. **Strong consistency** is non-negotiable for a file storage system -- use relational DB with ACID
3. **Long polling** is preferred over WebSocket for infrequent, server-initiated notifications
4. **Deduplication + cold storage + version limits** keep storage costs manageable at scale
5. **Conflict resolution** with first-write-wins and manual merge is pragmatic for multi-device sync

## See Also
- [[block-server]] | [[file-sync-and-conflict]] | [[metadata-database]]
- [[notification-service]] | [[storage-optimization]]